In [2]:
# Project Setup Cell – Install Dependencies first on T4GPU
!pip install transformers accelerate bitsandbytes --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.3 MB/s eta 0:00:00


In [3]:
# Project Setup Cell – Rebuild Architecture
# Create folders
!mkdir -p codebodh/analysis
!mkdir -p codebodh/mapping
!mkdir -p codebodh/engine
!mkdir -p codebodh/tutor

# --- Analyzer ---
with open("codebodh/analysis/analyzer.py", "w") as f:
    f.write("""
import ast
import traceback

def analyze_code(code: str):
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {
            "syntax_valid": False,
            "runtime_error": None,
            "error": {
                "type": "SyntaxError",
                "message": str(e),
                "line": e.lineno
            }
        }

    try:
        exec_globals = {}
        exec(code, exec_globals)
        return {
            "syntax_valid": True,
            "runtime_error": None,
            "error": None
        }
    except Exception as e:
        return {
            "syntax_valid": True,
            "runtime_error": True,
            "error": {
                "type": type(e).__name__,
                "message": str(e),
                "traceback": traceback.format_exc()
            }
        }
""")

# --- Concept Mapper ---
with open("codebodh/mapping/concept_mapper.py", "w") as f:
    f.write("""
ERROR_CONCEPT_MAP = {
    "NameError": {"concept": "Variable Scope", "subconcept": "Undefined Variable"},
    "TypeError": {"concept": "Data Types", "subconcept": "Type Mismatch"},
    "ZeroDivisionError": {"concept": "Arithmetic Operations", "subconcept": "Division by Zero"},
    "IndexError": {"concept": "Data Structures", "subconcept": "List Index Out of Range"},
    "SyntaxError": {"concept": "Syntax Rules", "subconcept": "Invalid Syntax"}
}

def map_error_to_concept(error_info: dict):
    if not error_info:
        return None

    error_type = error_info.get("type")

    if error_type in ERROR_CONCEPT_MAP:
        mapped = ERROR_CONCEPT_MAP[error_type]
        return {
            "concept": mapped["concept"],
            "subconcept": mapped["subconcept"],
            "confidence": 0.9
        }

    return {
        "concept": "Unknown",
        "subconcept": "Unmapped Error",
        "confidence": 0.3
    }
""")

# --- Diagnostic Engine ---
with open("codebodh/engine/diagnostic_engine.py", "w") as f:
    f.write("""
from analysis.analyzer import analyze_code
from mapping.concept_mapper import map_error_to_concept

def run_diagnostic(code: str):
    analysis_result = analyze_code(code)
    concept_result = None

    if analysis_result["error"]:
        concept_result = map_error_to_concept(analysis_result["error"])

    return {
        "analysis": analysis_result,
        "concept": concept_result
    }
""")

# --- Prompt Builder ---
with open("codebodh/tutor/prompt_builder.py", "w") as f:
    f.write("""
def build_pedagogical_prompt(diagnostic_output: dict):
    analysis = diagnostic_output.get("analysis")
    concept = diagnostic_output.get("concept")

    if not concept:
        return "No conceptual error detected."

    error_type = analysis["error"]["type"]
    error_message = analysis["error"]["message"]

    prompt = f\"\"\"
You are a disciplined programming tutor.

Student encountered:
Error Type: {error_type}
Error Message: {error_message}

Concept:
Main Concept: {concept['concept']}
Subconcept: {concept['subconcept']}

You must respond in EXACTLY this format:

Concept Diagnosis:
<1-2 sentences explaining the conceptual issue>

Probing Questions:
1.
2.

Hint:
<short hint without giving full corrected code>

Reflection:
<one reflective question>

Do NOT include anything outside this format.
Stay strictly within programming context.
\"\"\"

    return prompt.strip()
""")

# --- LLM Generator ---
with open("codebodh/tutor/llm_generator.py", "w") as f:
    f.write("""
import torch

def generate_llm_response(prompt: str, model, tokenizer):
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.3,
            top_p=0.9,
            do_sample=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded.split("[/INST]")[-1].strip()

    return response
""")

print("Architecture rebuilt successfully.")


Architecture rebuilt successfully.


In [4]:
# Project Setup Cell – Load Mistral Model once only
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

print("Model ready on:", model.device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model ready on: cuda:0


In [5]:
#before running the demo cell because notebooks don't remember module paths
import sys
sys.path.append("/content/codebodh")

from engine.diagnostic_engine import run_diagnostic
from tutor.prompt_builder import build_pedagogical_prompt
from tutor.llm_generator import generate_llm_response

print("Pipeline ready.")



Pipeline ready.


In [6]:
# Live Demo – Division by Zero Case
code = """
if(x>5)
Print(x)
"""

diagnostic = run_diagnostic(code)
prompt = build_pedagogical_prompt(diagnostic)
response = generate_llm_response(prompt, model, tokenizer)

print("=== DIAGNOSTIC OUTPUT ===")
print(diagnostic)

print("\n=== TUTOR RESPONSE ===")
print(response)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


=== DIAGNOSTIC OUTPUT ===
{'analysis': {'syntax_valid': False, 'runtime_error': None, 'error': {'type': 'SyntaxError', 'message': "expected ':' (<unknown>, line 2)", 'line': 2}}, 'concept': {'concept': 'Syntax Rules', 'subconcept': 'Invalid Syntax', 'confidence': 0.9}}

=== TUTOR RESPONSE ===
Concept Diagnosis:
The error message indicates that there is a syntax error on line 2 of your code. This means that the Python interpreter encountered unexpected characters or missing elements in the code, such as a missing colon (:) that is required to indicate the start of a new block of code.

Probing Questions:
1. What is a colon (:) used for in Python?
2. Can you check line 2 of your code to see if a colon is missing?

Hint:
Make sure that each block of code is properly indented and that there is a colon at the end of each block to indicate the start of a new statement.

Reflection:
Have you double-checked that your code is properly formatted and that all blocks of code are started with a col